# Gemini Vision API com Output Estruturado Detalhado

Este notebook demonstra como usar o SDK `google-genai` para obter uma análise detalhada da imagem em formato JSON estruturado.

In [1]:
import os
import pathlib
import json
from typing import List
from google import genai
from pydantic import BaseModel
from dotenv import load_dotenv
from PIL import Image

# Carrega as variáveis de ambiente
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# Definição do novo esquema detalhado para o output estruturado
class ImageAnalysis(BaseModel):
    objeto_principal: str
    descricao_detalhada: str
    cores_predominantes: List[str]
    sentimento_da_imagem: str
    elementos_secundarios: List[str]
    localizacao_provavel: str

In [17]:
# Listagem de imagens
image_folder = pathlib.Path("images")
if not image_folder.exists():
    image_folder = pathlib.Path("imagens")

images = list(image_folder.glob("*"))
for i, img_path in enumerate(images):
    print(f"{i}: {img_path.name}")

0: bird.jpg
1: cats_dog.png
2: kitchen.png
3: pizza.png


### Execução com Análise Detalhada (JSON)
O Gemini preencherá todos os campos definidos na classe `ImageAnalysis`.

In [18]:
# Selecione o índice da imagem desejada
idx = 3 # Exemplo: pizza.png

if idx < len(images):
    img_path = images[idx]
    img = Image.open(img_path)
    
    print(f"Analisando: {img_path.name}")
    
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=[
            "Analise a imagem abaixo e forneça os detalhes de acordo com o esquema JSON solicitado.",
            img
        ],
        config={
            'response_mime_type': 'application/json',
            'response_schema': ImageAnalysis,
        }
    )
    
    # Parse automático
    resultado = response.parsed
    
    # Exibe o resultado como JSON formatado
    print("\nResultado da Análise Detalhada (JSON):")
    print(resultado.model_dump_json(indent=2))
else:
    print("Índice de imagem inválido.")

Analisando: pizza.png

Resultado da Análise Detalhada (JSON):
{
  "objeto_principal": "Pizza",
  "descricao_detalhada": "Uma pizza artesanal fatiada, apresentada com uma borda alta e dourada, levemente tostada. A cobertura consiste em um molho de tomate vibrante, queijo muçarela derretido, fatias de pepperoni e folhas de manjericão fresco. Ela está disposta sobre um papel manteiga em cima de uma tábua de madeira rústica.",
  "cores_predominantes": [
    "Vermelho",
    "Dourado",
    "Branco",
    "Verde",
    "Marrom"
  ],
  "sentimento_da_imagem": "Apetite e conforto rústico",
  "elementos_secundarios": [
    "Papel manteiga",
    "Tábua de madeira",
    "Folhas de manjericão frescas",
    "Migalhas de pão",
    "Superfície de madeira"
  ],
  "localizacao_provavel": "Cozinha doméstica ou restaurante"
}
